In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected")


In [ ]:


import importlib.metadata as md
import subprocess
import sys

packages = [
    "torch",
    "transformers",
    "datasets",
    "accelerate",
    "pandas",
    "numpy",
    "pyarrow",
    "fsspec",
    "gcsfs"
]

for package in packages:
    try:
        print(f"{package}: {md.version(package)}")
    except md.PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")

print("\nChecking dependency compatibility...")
result = subprocess.run(
    [sys.executable, "-m", "pip", "check"],
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)
print("Exit code:", result.returncode)


In [ ]:

%pip install -q "gcsfs==2025.3.0" "fsspec==2025.3.0" jedi

In [ ]:

%pip check

In [ ]:

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

MODEL_NAME = "google/muril-base-cased"

LABEL2ID = {
    "Against": 0,
    "Favour": 1,
    "None": 2
}

ID2LABEL = {v: k for k, v in LABEL2ID.items()}

print("Loading MuRIL tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=False
)

print("Tokenizer loaded successfully!")
print("Tokenizer type:", type(tokenizer).__name__)

print("\nLoading MuRIL classification model...")

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=ID2LABEL,
    label2id=LABEL2ID
)

print("Model loaded successfully!")
print("Model type:", type(model).__name__)
print("Number of labels:", model.config.num_labels)

In [ ]:

sample_texts = [
    "जलवायु परिवर्तन एक गंभीर समस्या है।",
    "জলবায়ু পরিবর্তন একটি গুরুতর সমস্যা।",
    "Climate change is a serious concern.",
    "Global warming बहुत serious problem है।"
]

for text in sample_texts:
    tokens = tokenizer.tokenize(text)

    print("\nOriginal:", text)
    print("Tokens:", tokens)
    print("Token count:", len(tokens))
    print("-" * 50)

In [ ]:

from google.colab import files

uploaded = files.upload()

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:

import pandas as pd

train_df = pd.read_csv(
    "train.csv",
    encoding="utf-8-sig",
    keep_default_na=False
)

val_df = pd.read_csv(
    "validation.csv",
    encoding="utf-8-sig",
    keep_default_na=False
)

print("Training shape:", train_df.shape)
print("Validation shape:", val_df.shape)

print("\nTraining labels:")
print(train_df["stance"].value_counts(dropna=False))

print("\nValidation labels:")
print(val_df["stance"].value_counts(dropna=False))

print("\nRequired columns present:")
for name, df in [("Train", train_df), ("Validation", val_df)]:
    required = {"comment_text", "stance"}
    print(name, required.issubset(df.columns))

In [ ]:

from datasets import Dataset

LABEL2ID = {
    "Against": 0,
    "Favour": 1,
    "None": 2
}

ID2LABEL = {v: k for k, v in LABEL2ID.items()}

def prepare_data(df):
    df = df[["comment_text", "stance"]].copy()

    df["comment_text"] = df["comment_text"].astype(str).str.strip()
    df["stance"] = df["stance"].astype(str).str.strip()

    assert df["stance"].isin(LABEL2ID).all(), \
        "Invalid stance label detected!"

    assert df["comment_text"].str.len().gt(0).all(), \
        "Empty comment detected!"

    df["labels"] = df["stance"].map(LABEL2ID).astype(int)

    return Dataset.from_pandas(
        df[["comment_text", "labels"]],
        preserve_index=False
    )

train_dataset = prepare_data(train_df)
val_dataset = prepare_data(val_df)

print("Train dataset:", len(train_dataset))
print("Validation dataset:", len(val_dataset))
print("First training label:", train_dataset[0]["labels"])

In [ ]:

MAX_LENGTH = 128

def tokenize_function(batch):
    return tokenizer(
        batch["comment_text"],
        truncation=True,
        max_length=MAX_LENGTH
    )

train_tokenized = train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["comment_text"]
)

val_tokenized = val_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["comment_text"]
)

print("Training samples:", len(train_tokenized))
print("Validation samples:", len(val_tokenized))

print("\nTokenized columns:")
print(train_tokenized.column_names)

print("\nFirst sample token count:")
print(len(train_tokenized[0]["input_ids"]))

In [ ]:

from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    padding=True,
    return_tensors="pt"
)

print("Dynamic padding collator ready!")

sample_batch = data_collator([
    train_tokenized[0],
    train_tokenized[1]
])

for key, value in sample_batch.items():
    print(key, value.shape)

In [ ]:

import numpy as np

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    f1_score
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = (
        precision_recall_fscore_support(
            labels,
            predictions,
            average="macro",
            zero_division=0
        )
    )

    return {
        "accuracy": accuracy_score(labels, predictions),
        "precision_macro": precision,
        "recall_macro": recall,
        "f1_macro": f1,
        "f1_micro": f1_score(
            labels, predictions,
            average="micro",
            zero_division=0
        ),
        "f1_weighted": f1_score(
            labels, predictions,
            average="weighted",
            zero_division=0
        )
    }

print("Evaluation metrics configured!")

In [ ]:

import inspect
import transformers
from transformers import TrainingArguments

params = inspect.signature(
    TrainingArguments.__init__
).parameters

print("Transformers:", transformers.__version__)

for name in [
    "eval_strategy",
    "evaluation_strategy",
    "save_strategy",
    "load_best_model_at_end",
    "metric_for_best_model",
    "fp16",
    "report_to"
]:
    print(f"{name}: {name in params}")

In [ ]:

from transformers import set_seed

SEED = 42
set_seed(SEED)

print("Random seed set to:", SEED)
print("GPU available:", torch.cuda.is_available())

In [ ]:

from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="/content/muril_standard",

    num_train_epochs=3,
    learning_rate=2e-5,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,

    fp16=False,
    seed=42,
    data_seed=42,

    save_total_limit=2,
    report_to="none"
)

print("TrainingArguments configured successfully!")
print("Epochs:", training_args.num_train_epochs)
print("Learning rate:", training_args.learning_rate)
print("Training batch size:",
      training_args.per_device_train_batch_size)
print("Evaluation strategy:", training_args.eval_strategy)
print("Best model metric:",
      training_args.metric_for_best_model)

In [ ]:

import inspect
from transformers import Trainer

trainer_params = inspect.signature(
    Trainer.__init__
).parameters

trainer_kwargs = {
    "model": model,
    "args": training_args,
    "train_dataset": train_tokenized,
    "eval_dataset": val_tokenized,
    "data_collator": data_collator,
    "compute_metrics": compute_metrics
}

# Support different Transformers versions
if "processing_class" in trainer_params:
    trainer_kwargs["processing_class"] = tokenizer
elif "tokenizer" in trainer_params:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = Trainer(**trainer_kwargs)

print("MuRIL Trainer initialized successfully!")
print("Model:", trainer.model.__class__.__name__)
print("Training samples:", len(trainer.train_dataset))
print("Validation samples:", len(trainer.eval_dataset))
print("Device:", trainer.args.device)

In [ ]:

import time

print("Starting standard MuRIL training...")
print("Device:", trainer.args.device)

start_time = time.time()

train_result = trainer.train()

end_time = time.time()

print("\nTraining completed!")
print(f"Training time: {(end_time - start_time) / 60:.2f} minutes")

print("\nTraining metrics:")
print(train_result.metrics)

In [ ]:

import numpy as np
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score
)

# Predict using the best selected checkpoint
pred_output = trainer.predict(val_tokenized)

predictions = np.argmax(
    pred_output.predictions,
    axis=-1
)

true_labels = np.array(val_tokenized["labels"])

label_names = ["Against", "Favour", "None"]
label_ids = [0, 1, 2]

print("STANDARD MuRIL VALIDATION RESULTS")
print("=" * 50)

print("\nClassification Report:")
print(
    classification_report(
        true_labels,
        predictions,
        labels=label_ids,
        target_names=label_names,
        zero_division=0,
        digits=4
    )
)

print("\nConfusion Matrix:")
cm = confusion_matrix(
    true_labels,
    predictions,
    labels=label_ids
)
print(cm)

print("\nPredicted Class Distribution:")
for label_id, label_name in enumerate(label_names):
    count = np.sum(predictions == label_id)
    print(f"{label_name}: {count}")

print("\nValidation Accuracy:",
      accuracy_score(true_labels, predictions))

print("Validation Macro-F1:",
      f1_score(
          true_labels,
          predictions,
          labels=label_ids,
          average="macro",
          zero_division=0
      ))

In [ ]:

from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

SAVE_DIR = Path(
    "/content/drive/MyDrive/IndicClimateStance/muril_standard"
)

SAVE_DIR.mkdir(parents=True, exist_ok=True)

print("Save directory:", SAVE_DIR)

In [ ]:

import json
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (
    classification_report,
    confusion_matrix
)

names = ["Against", "Favour", "None"]
ids = [0, 1, 2]

report = classification_report(
    true_labels,
    predictions,
    labels=ids,
    target_names=names,
    output_dict=True,
    zero_division=0
)

metrics = {
    "model": "google/muril-base-cased",
    "experiment": "standard",
    "evaluation_split": "validation",
    "epochs": 3,
    "learning_rate": 2e-5,
    "batch_size": 8,
    "seed": 42,
    "accuracy": float(report["accuracy"]),
    "precision_macro": float(report["macro avg"]["precision"]),
    "recall_macro": float(report["macro avg"]["recall"]),
    "f1_macro": float(report["macro avg"]["f1-score"]),
    "f1_weighted": float(report["weighted avg"]["f1-score"]),
    "f1_micro": float(report["accuracy"]),
    "classification_report": report,
    "predicted_distribution": {
        name: int((predictions == i).sum())
        for i, name in enumerate(names)
    }
}

with open(
    SAVE_DIR / "muril_metrics.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(metrics, f, indent=4)

cm = confusion_matrix(
    true_labels,
    predictions,
    labels=ids
)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap="Blues")

ax.set_xticks(range(3), names)
ax.set_yticks(range(3), names)
ax.set_xlabel("Predicted label")
ax.set_ylabel("True label")
ax.set_title("Standard MuRIL - Validation Confusion Matrix")

for i in range(3):
    for j in range(3):
        ax.text(
            j, i, str(cm[i, j]),
            ha="center", va="center",
            color="white" if cm[i, j] > cm.max()/2 else "black"
        )

fig.colorbar(im, ax=ax)
fig.tight_layout()

fig.savefig(
    SAVE_DIR / "muril_confusion_matrix.png",
    dpi=200
)
plt.show()
plt.close(fig)

print("Standard MuRIL results saved successfully!")
print("Files:")
for file in SAVE_DIR.iterdir():
    print("-", file.name)

In [ ]:

MODEL_SAVE_DIR = SAVE_DIR / "best_model"

trainer.save_model(str(MODEL_SAVE_DIR))
tokenizer.save_pretrained(str(MODEL_SAVE_DIR))

print("Best standard MuRIL model saved!")
print("Location:", MODEL_SAVE_DIR)

In [ ]:

import numpy as np
import torch

from sklearn.utils.class_weight import compute_class_weight

train_labels = np.array(
    train_dataset["labels"]
)

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1, 2]),
    y=train_labels
)

class_weights = torch.tensor(
    class_weights,
    dtype=torch.float32
)

print("Class weights:")
for i, name in ID2LABEL.items():
    print(f"{name}: {class_weights[i]:.4f}")

print("\nGPU:", torch.cuda.get_device_name(0))

In [ ]:

import inspect
import torch
import torch.nn as nn

from transformers import Trainer

class WeightedTrainer(Trainer):

    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)

        if class_weights is None:
            raise ValueError("class_weights must be provided")

        self.class_weights = class_weights.detach().clone()

    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        num_items_in_batch=None,
        **kwargs
    ):
        labels = inputs["labels"]

        model_inputs = {
            key: value
            for key, value in inputs.items()
            if key != "labels"
        }

        outputs = model(**model_inputs)
        logits = outputs.logits

        loss_fn = nn.CrossEntropyLoss(
            weight=self.class_weights.to(
                device=logits.device,
                dtype=torch.float32
            )
        )

        loss = loss_fn(
            logits.float().view(-1, 3),
            labels.view(-1)
        )

        return (loss, outputs) if return_outputs else loss


print("WeightedTrainer defined successfully!")

print(
    "Trainer compute_loss signature:",
    inspect.signature(Trainer.compute_loss)
)

In [ ]:

from transformers import (
    AutoModelForSequenceClassification,
    set_seed
)

set_seed(42)

weighted_model = AutoModelForSequenceClassification.from_pretrained(
    "google/muril-base-cased",
    num_labels=3,
    id2label=ID2LABEL,
    label2id=LABEL2ID
)

print("Fresh MuRIL model loaded successfully!")
print("Model:", weighted_model.__class__.__name__)
print("Number of labels:", weighted_model.config.num_labels)
print("GPU available:", torch.cuda.is_available())

In [ ]:

from transformers import TrainingArguments

weighted_training_args = TrainingArguments(
    output_dir="/content/muril_weighted",

    num_train_epochs=3,
    learning_rate=2e-5,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,

    fp16=False,
    seed=42,
    data_seed=42,

    save_total_limit=2,
    report_to="none"
)

print("Weighted training configuration ready!")
print("Epochs:", weighted_training_args.num_train_epochs)
print("Learning rate:", weighted_training_args.learning_rate)
print("Batch size:", weighted_training_args.per_device_train_batch_size)
print("Best checkpoint metric:",
      weighted_training_args.metric_for_best_model)

In [ ]:

import inspect
from transformers import Trainer

weighted_trainer_params = inspect.signature(
    WeightedTrainer.__init__
).parameters

weighted_trainer_kwargs = {
    "model": weighted_model,
    "args": weighted_training_args,
    "train_dataset": train_tokenized,
    "eval_dataset": val_tokenized,
    "data_collator": data_collator,
    "compute_metrics": compute_metrics,
    "class_weights": class_weights
}

# Check the underlying Trainer API for tokenizer compatibility
trainer_params = inspect.signature(
    Trainer.__init__
).parameters

if "processing_class" in trainer_params:
    weighted_trainer_kwargs["processing_class"] = tokenizer
elif "tokenizer" in trainer_params:
    weighted_trainer_kwargs["tokenizer"] = tokenizer

weighted_trainer = WeightedTrainer(
    **weighted_trainer_kwargs
)

print("WeightedTrainer initialized successfully!")
print("Model:", weighted_trainer.model.__class__.__name__)
print("Training samples:", len(weighted_trainer.train_dataset))
print("Validation samples:", len(weighted_trainer.eval_dataset))
print("Device:", weighted_trainer.args.device)

print("\nClass weights:")
for i, name in ID2LABEL.items():
    print(f"{name}: {class_weights[i]:.4f}")

In [ ]:

import time

print("Starting Weighted MuRIL training...")
print("Device:", weighted_trainer.args.device)

start_time = time.time()

weighted_train_result = weighted_trainer.train()

end_time = time.time()

print("\nWeighted MuRIL training completed!")
print(
    f"Training time: {(end_time - start_time) / 60:.2f} minutes"
)

print("\nTraining metrics:")
print(weighted_train_result.metrics)

print("\nBest checkpoint:")
print(weighted_trainer.state.best_model_checkpoint)

print("\nBest validation Macro-F1:")
print(weighted_trainer.state.best_metric)

In [ ]:

import numpy as np
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score
)

# Predict using the best weighted MuRIL checkpoint
weighted_pred_output = weighted_trainer.predict(
    val_tokenized
)

weighted_predictions = np.argmax(
    weighted_pred_output.predictions,
    axis=-1
)

weighted_true_labels = np.array(
    val_tokenized["labels"]
)

label_names = ["Against", "Favour", "None"]
label_ids = [0, 1, 2]

print("WEIGHTED MuRIL VALIDATION RESULTS")
print("=" * 50)

print("\nClassification Report:")
print(
    classification_report(
        weighted_true_labels,
        weighted_predictions,
        labels=label_ids,
        target_names=label_names,
        zero_division=0,
        digits=4
    )
)

print("\nConfusion Matrix:")
weighted_cm = confusion_matrix(
    weighted_true_labels,
    weighted_predictions,
    labels=label_ids
)
print(weighted_cm)

print("\nPredicted Class Distribution:")
for label_id, label_name in enumerate(label_names):
    count = np.sum(weighted_predictions == label_id)
    print(f"{label_name}: {count}")

print("\nValidation Accuracy:",
      accuracy_score(
          weighted_true_labels,
          weighted_predictions
      ))

print("Validation Macro-F1:",
      f1_score(
          weighted_true_labels,
          weighted_predictions,
          labels=label_ids,
          average="macro",
          zero_division=0
      ))

print("\nBest checkpoint:",
      weighted_trainer.state.best_model_checkpoint)

In [ ]:

import json
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score
)

WEIGHTED_SAVE_DIR = Path(
    "/content/drive/MyDrive/IndicClimateStance/muril_weighted"
)

WEIGHTED_SAVE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

names = ["Against", "Favour", "None"]
ids = [0, 1, 2]

report = classification_report(
    weighted_true_labels,
    weighted_predictions,
    labels=ids,
    target_names=names,
    output_dict=True,
    zero_division=0
)

metrics = {
    "model": "google/muril-base-cased",
    "experiment": "weighted",
    "evaluation_split": "validation",
    "epochs": 3,
    "learning_rate": 2e-5,
    "batch_size": 8,
    "seed": 42,
    "best_checkpoint": weighted_trainer.state.best_model_checkpoint,
    "best_epoch": 2,
    "accuracy": float(report["accuracy"]),
    "precision_macro": float(report["macro avg"]["precision"]),
    "recall_macro": float(report["macro avg"]["recall"]),
    "f1_macro": float(report["macro avg"]["f1-score"]),
    "f1_micro": float(
        f1_score(
            weighted_true_labels,
            weighted_predictions,
            labels=ids,
            average="micro",
            zero_division=0
        )
    ),
    "f1_weighted": float(report["weighted avg"]["f1-score"]),
    "classification_report": report,
    "class_weights": {
        name: float(class_weights[i])
        for i, name in enumerate(names)
    },
    "predicted_distribution": {
        name: int((weighted_predictions == i).sum())
        for i, name in enumerate(names)
    }
}

with open(
    WEIGHTED_SAVE_DIR / "muril_weighted_metrics.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(metrics, f, indent=4)

cm = confusion_matrix(
    weighted_true_labels,
    weighted_predictions,
    labels=ids
)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap="Blues")

ax.set_xticks(range(3), names)
ax.set_yticks(range(3), names)
ax.set_xlabel("Predicted label")
ax.set_ylabel("True label")
ax.set_title("Weighted MuRIL - Validation Confusion Matrix")

for i in range(3):
    for j in range(3):
        ax.text(
            j, i, str(cm[i, j]),
            ha="center",
            va="center",
            color="white" if cm[i, j] > cm.max()/2 else "black"
        )

fig.colorbar(im, ax=ax)
fig.tight_layout()

fig.savefig(
    WEIGHTED_SAVE_DIR / "muril_weighted_confusion_matrix.png",
    dpi=200
)

plt.show()
plt.close(fig)

print("Weighted MuRIL results saved!")
print("\nSaved files:")

for file in WEIGHTED_SAVE_DIR.iterdir():
    print("-", file.name)

In [ ]:

WEIGHTED_MODEL_DIR = WEIGHTED_SAVE_DIR / "best_model"

weighted_trainer.save_model(
    str(WEIGHTED_MODEL_DIR)
)

tokenizer.save_pretrained(
    str(WEIGHTED_MODEL_DIR)
)

print("Best weighted MuRIL model saved!")
print("Location:", WEIGHTED_MODEL_DIR)

In [ ]:

import json
import pandas as pd
from pathlib import Path

BASE_DIR = Path(
    "/content/drive/MyDrive/IndicClimateStance"
)

standard_path = (
    BASE_DIR / "muril_standard" / "muril_metrics.json"
)

weighted_path = (
    BASE_DIR / "muril_weighted" / "muril_weighted_metrics.json"
)

with open(standard_path, encoding="utf-8") as f:
    standard = json.load(f)

with open(weighted_path, encoding="utf-8") as f:
    weighted = json.load(f)

metric_names = [
    "accuracy",
    "precision_macro",
    "recall_macro",
    "f1_macro",
    "f1_micro",
    "f1_weighted"
]

rows = []

for experiment, data in [
    ("Standard MuRIL", standard),
    ("Weighted MuRIL", weighted)
]:
    row = {
        "model": experiment,
        "evaluation_split": data["evaluation_split"]
    }

    for metric in metric_names:
        row[metric] = data[metric]

    rows.append(row)

comparison_df = pd.DataFrame(rows)

OUTPUT_PATH = BASE_DIR / "muril_comparison.csv"

comparison_df.to_csv(
    OUTPUT_PATH,
    index=False
)

print("MuRIL comparison saved successfully!")
print("Location:", OUTPUT_PATH)

display(comparison_df.round(4))